In [1]:
import os
import sys
import random
from collections import deque

import matplotlib.pyplot as plt

import numpy as np
from scipy.ndimage import zoom

import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

2026-03-12 08:31:39.660357: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 08:31:41.802833: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773300702.594778      97 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773300702.792913      97 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773300704.607838      97 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
# git clone https://github.com/MIC-DKFZ/nnInteractive
# cd nnInteractive
# pip install -e .

In [3]:
os.chdir("/nfs/norasys/notebooks/machauep")

In [4]:
from nnInteractive.inference.inference_session import nnInteractiveInferenceSession
from Helpers import Helpers
Helpers = Helpers()

nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.


In [5]:
import os
import torch
import SimpleITK as sitk
from huggingface_hub import snapshot_download  # Install huggingface_hub if not already installed

# Data

In [6]:
sys.path.append("/software")
sys.path.append(os.environ['DPXROOT'] + '/src/python')
import patchwork_dev.patchwork as patchwork

from DPX_core import *


project = os.environ['DPXproject']
ext = lambda d,e: { (k+e):d[k] for k in d }
ximg = {}
limg = {}
def get(ids,img,label):
    global ximg,limg
    ximg.update(ext(DPX_selectFiles(project, [ids, img]), img))
    limg.update(ext(DPX_selectFiles(project, [ids, label]), img))

# --- Body Data ---
roi = '3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSED*'
img = 'wholebody/wbcomp2.nii.gz'
patients = ['104177', '104178']

for p in patients:
    get(p,roi,img) 
    
# --- Brain Data ---
# get('113*','T1_crop.nii','freesurfer/wmparc_atlas.nii.gz')
# get('113*','T1km_crop.nii','freesurfer/wmparc_atlas.nii.gz')
# get('113*','T2_crop.nii','freesurfer/wmparc_atlas.nii.gz')

getfname = lambda img: [ {k:img[k][list(img[k].keys())[0]]['FilePath'] for k in img} ]
getidx = lambda img: [ {k:img[k][list(img[k].keys())[0]]['STag']+k.split("#")[1] for k in img} ]

with tf.device("/cpu:0"):                

    loading = {  "integer_labels":True  }
    contrasts= getfname(ximg)
    tidx= getidx(ximg)[0]
    labels= getfname(limg)
    subjects = list(contrasts[0].keys())
    tset_,lset_,rset_,subjs = patchwork.improc_utils.load_data_structured(
                               contrasts=contrasts,labels=labels,subjects=subjects, **loading)

    tset = []
    lset = []
    rset = []
    iset = []
    didx = {}
    cnt = 0
    for k in range(len(tset_)):
        if tidx[subjs[k]] not in didx:
            didx[tidx[subjs[k]]] = str(cnt)
            cnt = cnt + 1
        for j in range(tset_[k].shape[4]):
            tset.append(tset_[k][...,j:j+1])
            tmp_ = lset_[k][:,:,:,:,0:1]
            tmp_ = tf.where(tmp_<0,0,tmp_)
            lset.append(tf.cast(tmp_,dtype=tf.int32))
            rset.append(rset_[k])
            iset.append( didx[tidx[subjs[k]]]+"_"+str(j) )
        

    iset_ = dict([(y,x+1) for x,y in enumerate(sorted(set(iset)))])
    iset = [iset_[x] for x in iset]

going to load 2 items
X

I0000 00:00:1773300776.856151      97 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46569 MB memory:  -> device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:2a:00.0, compute capability: 8.9


X


# Functions

In [7]:
def generate_initial_prompt(seg_3d, visualize=False, min_pixels=50):
    """
    Generates an initial prompt from a 3D segmentation array by selecting 
    a random ROI (label) and a central slice along a random axis.
    Ensures the selected ROI in the slice has at least `min_pixels`.

    Parameters
    ----------
    seg_3d : np.ndarray
        3D numpy array representing the segmentation (each integer = different ROI).
    visualize : bool, optional
        If True, display the selected slice (default: False).
    min_pixels : int, optional
        Minimum number of pixels for an ROI in a slice to be considered (default: 50).

    Returns
    -------
    initial_prompt_3d : np.ndarray
        3D array of same shape as `seg_3d`, containing only one 2D slice with the selected ROI.
    initial_prompt_2d : np.ndarray
        The selected 2D slice mask.
    (axis, rand_i) : tuple
        Axis and slice index.
    selected_roi : int
        The chosen ROI label.
    """
    seg_3d = np.asarray(seg_3d)
    
    def get_valid_slices(seg_3d, axis):
        non_empty_indices = []
        for i in range(seg_3d.shape[axis]):
            slice_ = np.take(seg_3d, i, axis=axis)
            if np.any(slice_ != 0):
                non_empty_indices.append(i)
        return non_empty_indices

    axis = random.choice([0, 1, 2])
    valid_indices = get_valid_slices(seg_3d, axis)
    
    if not valid_indices:
        raise ValueError("No non-empty slices found in segmentation.")

    attempt = 0
    while True:
        attempt += 1
        if attempt > 50:
            raise ValueError("Could not find ROI above min_pixels after 50 attempts.")

        rand_i = random.choice(valid_indices)
        rand_slice = np.take(seg_3d, rand_i, axis=axis)

        rand_slice_1d = tf.reshape(rand_slice, shape=(-1,))
        unique_labels, _ = tf.unique(rand_slice_1d)
        unique_labels = tf.boolean_mask(unique_labels, unique_labels != 0)

        if len(unique_labels) == 0:
            continue

        # Filter ROIs by pixel count
        eligible_labels = []
        for label in unique_labels.numpy():
            pixel_count = np.sum(rand_slice == label)
            if pixel_count >= min_pixels:
                eligible_labels.append(label)

        if not eligible_labels:
            continue  # try again if no eligible ROI found in this slice

        selected_roi = random.choice(eligible_labels)
        break

    roi_mask = tf.cast((rand_slice == selected_roi), tf.float32)
    initial_prompt_3d = np.zeros_like(seg_3d)

    if axis == 0:
        initial_prompt_3d[rand_i, :, :] = roi_mask
        initial_prompt_2d = roi_mask
    elif axis == 1:
        initial_prompt_3d[:, rand_i, :] = roi_mask
        initial_prompt_2d = roi_mask
    else:
        initial_prompt_3d[:, :, rand_i] = roi_mask
        initial_prompt_2d = roi_mask

    if visualize:
        plt.imshow(initial_prompt_2d)
        plt.title(f"ROI {selected_roi} | Axis {axis}, Slice {rand_i} | {int(tf.reduce_sum(roi_mask))} px")
        plt.show()

    return initial_prompt_3d, initial_prompt_2d, (axis, rand_i), selected_roi

In [8]:
from tensorflow.image import ssim

def compare_ssim(tensor_1, tensor_2, max_val=1.0):
    """
    SSIM comparison for two normalized 128x128 tensors.
    """ 
    if len(tensor_1.shape) == 2:
        tensor_1 = tensor_1[..., tf.newaxis]
    if len(tensor_2.shape) == 2:
        tensor_2 = tensor_2[..., tf.newaxis]
    
    ssim_score = ssim(tensor_1[tf.newaxis, ...], 
                     tensor_2[tf.newaxis, ...], 
                     max_val=max_val)
    
    ssim_score = tf.squeeze(ssim_score)
    return ssim_score.numpy()

In [9]:
def dice_window_nn(y_true_3d, y_pred_3d, axis, center_idx, window=10, smooth=1e-6):
    # Move the chosen axis to the first dimension
    y_true_3d = tf.transpose(y_true_3d, perm=[axis] + [i for i in range(3) if i != axis])
    y_pred_3d = tf.transpose(y_pred_3d, perm=[axis] + [i for i in range(3) if i != axis])

    num_slices = tf.shape(y_true_3d)[0]
    
    # Calculate window bounds
    start = tf.maximum(center_idx - window, 0)
    end = tf.minimum(center_idx + window + 1, num_slices)
    
    # Extract slices in the window
    y_true_window = y_true_3d[start:end]
    y_pred_window = y_pred_3d[start:end]
    
    # Vectorized Dice computation
    def compute_dice_for_slice(i):
        return Helpers.dice_score_tf(y_true_window[i], y_pred_window[i], smooth)
    
    # Compute Dice for all slices in parallel
    indices = tf.range(tf.shape(y_true_window)[0])
    dice_scores = tf.map_fn(compute_dice_for_slice, indices, dtype=tf.float32)
    
    return tf.reduce_mean(dice_scores).numpy()

def dice_window_prompt(y_true_3d, y_pred_3d, forward_idxs, window=10, smooth=1e-6):
    num_slices = tf.shape(y_true_3d)[0]
    center_idx = (num_slices - len(forward_idxs)) - 1
    
    # Calculate window bounds
    start = tf.maximum(center_idx - window, 0)
    end = tf.minimum(center_idx + window + 1, num_slices)
    
    # Extract slices in the window
    y_true_window = y_true_3d[start:end]
    y_pred_window = y_pred_3d[start:end]
    
    # Vectorized Dice computation
    def compute_dice_for_slice(i):
        return Helpers.dice_score_tf(y_true_window[i], y_pred_window[i], smooth)
    
    # Compute Dice for all slices in parallel
    indices = tf.range(tf.shape(y_true_window)[0])
    dice_scores = tf.map_fn(compute_dice_for_slice, indices, dtype=tf.float32)
    
    # Return average
    return tf.reduce_mean(dice_scores).numpy()

In [10]:
def volumetric_dice(y_true, y_pred, smooth=1e-6):
    """
    Compute Dice over the entire 3D volume.
    y_true, y_pred: binary arrays with identical shapes [D,H,W] (or any shape)
    Returns single float (1.0 if both empty).
    """
    if isinstance(y_true, tf.Tensor):
        y_true = y_true.numpy()
    if isinstance(y_pred, tf.Tensor):
        y_pred = y_pred.numpy()
        
    # ensure boolean
    A = (y_true != 0).astype(np.bool_)
    B = (y_pred != 0).astype(np.bool_)
    inter = np.sum(A & B)
    sumA = np.sum(A)
    sumB = np.sum(B)
    if sumA == 0 and sumB == 0:
        return 1.0
    dice = (2.0 * inter + smooth) / (sumA + sumB + smooth)
    return float(dice)

# Functions Prompt U-Net

In [11]:
def extract_valid_slices(seg_3d, prompt_axis):
    nonempty_mask = []

    for i in range(seg_3d.shape[prompt_axis]):
        # Extract the 2D slice along the prompt_axis
        slice_2d = np.take(seg_3d, i, axis=prompt_axis)
        if np.any(slice_2d):  # equivalent to len(np.nonzero(slice_2d)[0]) > 0
            nonempty_mask.append(True)
        else:
            nonempty_mask.append(False)

    nonempty_mask = np.array(nonempty_mask)

    # Find valid slice indices
    valid_indices = np.where(nonempty_mask)[0]

    # Limit backward and forward search to only valid (non-empty) slices
    backward_indices = [i for i in range(prompt_idx - 1, -1, -1) if i in valid_indices]
    forward_indices  = [i for i in range(prompt_idx + 1, seg_3d.shape[prompt_axis]) if i in valid_indices]

    return backward_indices, forward_indices

In [22]:
def run_prompt_unet(img_3d, seg_3d, initial_prompt_2d_seg, prompt_axis, prompt_idx, reduce_mean, sim_diff_threshold=0.2, output_threshold=0.45, model='Prompt U-Net V1_9_3.keras', user_interactions=False, plotting=False): 
    # print(f'Model: {model}')
    loaded_model = tf.keras.models.load_model(model)
    
    # shape: (x,y,z)
    img_3d = np.squeeze(img_3d)
    
    initial_prompt_img = np.take(img_3d, prompt_idx, axis=prompt_axis)
    initial_prompt_img = Helpers.min_max_norm(initial_prompt_img) # MinMax only on img, not segmentation!
    
    initial_prompt_img = np.expand_dims(initial_prompt_img, axis=-1)
    initial_prompt_2d_seg = np.expand_dims(initial_prompt_2d_seg, axis=-1)
    initial_prompt_2d_seg = tf.cast(initial_prompt_2d_seg, tf.float32)
    initial_prompt_img = Helpers.shaping(initial_prompt_img)
    initial_prompt_2d_seg = Helpers.shaping(initial_prompt_2d_seg, binary=True)
    p = tf.concat([initial_prompt_img, initial_prompt_2d_seg], axis=-1)
    
    backward_indices, forward_indices = extract_valid_slices(seg_3d, prompt_axis)
    indices = backward_indices + forward_indices
    
    results, ground_truths, user_interacts_idx = [], [], []
    start_ssim = None
    
    buffer = deque(maxlen=6)
    buffer.append(initial_prompt_2d_seg)
    
    for i in indices:
        # print(i)
        
        # Ground Truth
        y = np.take(seg_3d, i, axis=prompt_axis)
        y = tf.cast(y, tf.float32)
        y = tf.expand_dims(y, axis=0)
        y = Helpers.shaping(y, binary=True)
        ground_truths.append(y)
        
        current_slice = np.take(img_3d, i, axis=prompt_axis)
        current_slice = np.expand_dims(current_slice, axis=-1)
        current_slice = Helpers.min_max_norm(current_slice)
        current_slice = Helpers.shaping(current_slice)
        
        pred = loaded_model.predict([current_slice, p], verbose=0)
        pred = tf.where(pred < output_threshold, 0.0, pred)
        pred = tf.where(pred >= output_threshold, 1.0, pred)
        
        buffer.append(pred)
        results.append(pred)
        
        if plotting:
            fig, axes = plt.subplots(1, 3, figsize=(6, 3)) 
            fig.suptitle(f'Slice {i}', fontsize=13)
            axes[0].imshow(current_slice[0,...,0] + y[0,...,0])
            axes[0].set_title('Input & GT')
            axes[0].axis('off')
            axes[1].imshow(p[0,...,0] + p[0,...,1])
            axes[1].set_title('Prompt')
            axes[1].axis('off')
            axes[2].imshow(pred[0,...,0])
            axes[2].set_title('Prediction')
            axes[2].axis('off')
            plt.tight_layout()
            plt.show()
            
        current_dice = Helpers.dice_score_tf(y, pred)
        # print(f'Dice: {current_dice:.3f}')
        
        # Update with GT
        if user_interactions and current_dice < 0.65:
            p = tf.concat([current_slice, y], axis=-1)
            user_interacts_idx.append(i)
            results[-1] = y # delete newest prediction and replace with user interaction

        current_ssim = compare_ssim(current_slice, p[...,0:1])
        # print(f'SSIM similarity: {current_ssim:.3f}')

        if start_ssim == None: 
            start_ssim = current_ssim

        diff = start_ssim - current_ssim
        # print("SSIM Diff:", diff)
        if  diff >= sim_diff_threshold and diff < 1.0 and sim_diff_threshold != 0:
            # print("Aktualisiert! Im nächsten run ist der Prompt neu!")
            if reduce_mean:
                p_y = tf.reduce_mean(tf.stack(list(buffer), axis=0), axis=0) # Min is more confident than mean
            else:
                p_y = tf.reduce_min(tf.stack(list(buffer), axis=0), axis=0) # Min is more confident than mean
            p_y = tf.math.sign(p_y)
            p = tf.concat([current_slice, p_y], axis=-1) # Current img and mean label from predictions
            start_ssim = None #Aktualisierung des start_ssim erst einen schritt später. ist das gut?
            buffer.clear()
            buffer.append(p_y)
    
        
        # Refresh p in the middle
        if len(backward_indices) > 0: # Rare case that we only have indixes right of the prompt
            if i == backward_indices[-1]: # backward_indices[-1] == one before inital prompt
                # print("refreshed p")   
                
                # Inverse all results that were added backwards before adding the prompt!
                results = list(reversed(results))
                ground_truths = list(reversed(ground_truths))

                results.append(initial_prompt_2d_seg)
                ground_truths.append(initial_prompt_2d_seg)
                p = tf.concat([initial_prompt_img, initial_prompt_2d_seg], axis=-1)
                buffer.clear()
                buffer.append(initial_prompt_2d_seg)
                
        # print()
    
    # Initial Prompt still need to appended even if there are no backward_indices
    if len(backward_indices) <= 0:
        results.append(initial_prompt_2d_seg)
        ground_truths.append(initial_prompt_2d_seg)
    
    gt_tensor = tf.stack(ground_truths, axis=0)
    results_tensor = tf.stack(results, axis=0)
    
    num_user_interacts = len(user_interacts_idx) + 1
    
    return np.squeeze(results_tensor), np.squeeze(gt_tensor), num_user_interacts, user_interacts_idx, backward_indices, forward_indices

# Main

Don't forget to set number of runs per Volume and user_interacts

In [23]:
result_dict = {}
runs_per_vol = 12
user_interacts = False
model = "prompt_unet/prompt_unetV2_1.keras"

# sim_diff_threshold = 0.4
# sim_diff_threshold = 0.5
sim_diff_threshold = 0.6

for e, img_3d in enumerate(tset):
    # print(e)
    img_3d = img_3d[0,:,:,:,0]
    seg_3d_original = tf.cast(lset[e][0,:,:,:,0], tf.float32)
    
    img_3d = img_3d.numpy()
    seg_3d_original = seg_3d_original.numpy()
    img_3d = np.expand_dims(img_3d, axis=0) # Ready for nnInteractive
    
    volume_key = "Volume " + str(e)
    result_dict[volume_key] = {}
    
    # How many random prompts/runs per Volume
    for num_runs in range(runs_per_vol):
        run_key = "Run " + str(num_runs)
        result_dict[volume_key][run_key] = {}
        
        # --- Get random inital Prompt ---
        initial_prompt_3d, initial_prompt_2d_seg, (prompt_axis, prompt_idx), selected_roi = generate_initial_prompt(seg_3d_original, visualize=False)
        seg_3d = seg_3d_original == selected_roi
        
        %matplotlib inline
#         plt.imshow(np.take(seg_3d, prompt_idx, prompt_axis) * 900 + np.take(np.squeeze(img_3d), prompt_idx, prompt_axis))
#         plt.show()
#         plt.imshow(initial_prompt_2d_seg)
#         plt.show()
        
        # In your main loop, add detailed visualization
        # fig, axes = plt.subplots(2, 3, figsize=(12, 8))
        # axes[0,0].imshow(np.take(seg_3d, prompt_idx, prompt_axis))
        # axes[0,0].set_title('GT at prompt slice')
        # axes[0,1].imshow(initial_prompt_2d_seg)
        # axes[0,1].set_title('Initial prompt')
        # axes[0,2].imshow(np.take(np.squeeze(img_3d), prompt_idx, prompt_axis))
        # axes[0,2].set_title('Image at prompt slice')

        
        # --- Prompt U-Net min ---
        results_p_unet, gt_tensor, num_user_interacts, user_interacts_idx, backward_indices, forward_indices = run_prompt_unet(
            img_3d, seg_3d, initial_prompt_2d_seg, prompt_axis, prompt_idx, reduce_mean = False, sim_diff_threshold=sim_diff_threshold, output_threshold=0.45, 
            model=model, user_interactions=user_interacts, plotting=False)
        
        
        # --- Prompt U-Net mean ---
        results_p_unet_mean, gt_tensor_mean, num_user_interacts_mean, user_interacts_idx_mean, backward_indices_mean, forward_indices_mean = run_prompt_unet(
            img_3d, seg_3d, initial_prompt_2d_seg, prompt_axis, prompt_idx, reduce_mean = True, sim_diff_threshold=sim_diff_threshold, output_threshold=0.45, 
            model=model, user_interactions=user_interacts, plotting=False)
        
        
#         print(f"Selected ROI: {selected_roi}")
#         print(f"Unique labels in slice: {np.unique(np.take(seg_3d_original, prompt_idx, prompt_axis))}")
#         print(f"ROI mask non-zero pixels: {np.count_nonzero(initial_prompt_2d_seg)}")

#         seg_3d_binary = seg_3d_original == selected_roi
#         print(f"Binary seg non-zero voxels: {np.count_nonzero(seg_3d_binary)}")
        
        # # Show some results if available
        # if 'results_p_unet' in locals():
        #     axes[1,0].imshow(results_p_unet[0] if results_p_unet.ndim == 3 else results_p_unet)
        #     axes[1,0].set_title('Prompt U-Net result')
        #     axes[1,1].imshow(gt_tensor[0] if gt_tensor.ndim == 3 else gt_tensor)
        #     axes[1,1].set_title('GT')
        # plt.show()
        
        vol_dice_p_unet = volumetric_dice(gt_tensor, results_p_unet)
        window_dice_p_unet = dice_window_prompt(gt_tensor, results_p_unet, forward_indices, window=10)
        
        vol_dice_p_unet_mean = volumetric_dice(gt_tensor_mean, results_p_unet_mean)
        window_dice_p_unet_mean = dice_window_prompt(gt_tensor_mean, results_p_unet_mean, forward_indices_mean, window=10)
        

        
        # --- Store results and information--- 
        result_dict[volume_key][run_key]["Prompt U-Net min"] = {}
        result_dict[volume_key][run_key]["Prompt U-Net mean"] = {}
        
        result_dict[volume_key][run_key]["Prompt U-Net min"]["Dices"] = (vol_dice_p_unet, window_dice_p_unet)
        result_dict[volume_key][run_key]["Prompt U-Net mean"]["Dices"] = (vol_dice_p_unet_mean, window_dice_p_unet_mean)
        
        # # The same for Prompt U-Net and nnInteractive
        # result_dict[volume_key][run_key]["Prompt U-Net min"]["User Interactions"] = num_user_interacts
        # result_dict[volume_key][run_key]["Prompt U-Net min"]["Shape Original Volume"] = initial_prompt_3d.shape
        # result_dict[volume_key][run_key]["Prompt U-Net min"]["Len Cropped Volume"] = len(backward_indices + forward_indices)
        # result_dict[volume_key][run_key]["Prompt U-Net min"]["Initial Prompt"] = (prompt_axis, prompt_idx)

Don't forget to set file name

In [24]:
file_name = "results_3d_body_no_interactions_21_combined_reduce_mean_3.pkl"

In [25]:
import pickle

with open(file_name, "wb") as f:
    pickle.dump(result_dict, f)